# 03 — Vector Search

*Level 1 — Naive RAG*

## Objective
Build a tiny in-memory vector index by hand and run Top-K search against it — no framework, ~10 lines of real logic.

See [`../theory/vector_search.md`](../theory/vector_search.md).


In [1]:
import sys
from pathlib import Path

# Make the local `src` package importable from notebooks/
sys.path.insert(0, str(Path.cwd().parent))


In [2]:
from src.embed import get_embedder
from src.retrieve import InMemoryVectorStore
from src.schema import Chunk, EmbeddedChunk

embedder = get_embedder()
store = InMemoryVectorStore()


## Index a handful of one-line facts


In [3]:
facts = [
    "Refunds are processed within 30 days of purchase.",
    "Enterprise refunds require account manager approval.",
    "New employees have a 90-day probation period.",
    "Employees accrue 15 days of PTO per year.",
    "The product supports offline mode with automatic sync.",
]

vectors = embedder.embed(facts)
embedded_chunks = [
    EmbeddedChunk(
        chunk=Chunk(id=f"fact-{i}", text=text, document_id=f"fact-{i}", position=0),
        vector=vector,
    )
    for i, (text, vector) in enumerate(zip(facts, vectors))
]
store.add(embedded_chunks)
print(f"Indexed {len(store)} facts.")


Indexed 5 facts.


## Search


In [4]:
def search(query, top_k=3):
    query_vector = embedder.embed_one(query)
    results = store.search(query_vector, top_k=top_k)
    for r in results:
        print(f"{r.score:.3f}  {r.chunk.text}")

search("How long is the trial period for a new hire?")


0.734  New employees have a 90-day probation period.
0.521  Refunds are processed within 30 days of purchase.
0.511  Employees accrue 15 days of PTO per year.


In [5]:
search("How much vacation time do I get?")


0.543  Employees accrue 15 days of PTO per year.
0.485  Refunds are processed within 30 days of purchase.
0.472  New employees have a 90-day probation period.


## Exercise

1. Add 2-3 facts of your own, re-index, and query for them.
2. Ask a question with **no** good match in the index (e.g. "What's the weather today?") — look at the scores. Notice brute-force search *always* returns Top-K, even when nothing is actually relevant — the score is your only signal, and Level 1 does nothing with it yet. (Level 4's Corrective RAG addresses this directly.)
3. Persist the index with `store.save("/tmp/my_index")` and reload it with `InMemoryVectorStore.load("/tmp/my_index")`.
